# Module 31 — torchao: Quantization & Sparsity for Production

> **PyTorch Architecture Optimization** — Make models faster and smaller with quantization, sparsity, and dtype optimization.

| | |
|---|---|
| **Prerequisites** | [Module 07 (Training)](../07_training/), [Module 08 (torch.compile)](../08_torch_compile/), [Module 29 (Mixed Precision)](../29_mixed_precision/) |
| **Time** | ~60 minutes |
| **GPU Required** | No (CPU-only, GPU-specific notes where applicable) |

---

## 1. What is Quantization?

Quantization maps continuous floating-point values to a smaller set of discrete values, reducing memory and enabling faster computation.

### The Math

```
Quantize:     q = round(x / scale) + zero_point
Dequantize:   x_approx = (q - zero_point) * scale
```

Where:
- `scale = (x_max - x_min) / (2^bits - 1)`
- `zero_point` maps real zero to an integer value

### Visual: FP32 → INT8 mapping

```
FP32:  -1.0 ──── -0.5 ──── 0.0 ──── 0.5 ──── 1.0  (continuous)
         │          │        │         │         │
         ▼          ▼        ▼         ▼         ▼
INT8:  -128 ─────  -64 ──── 0 ────── 64 ────── 127  (256 discrete levels)
```

The precision loss from this mapping is called **quantization error**.

In [ ]:
import torch
import torch.nn as nn
import time
import copy

print(f"PyTorch version: {torch.__version__}")

# Check torchao availability
try:
    import torchao
    print(f"torchao version: {torchao.__version__}")
    TORCHAO_AVAILABLE = True
except ImportError:
    print("torchao not installed — all fundamentals still work with pure PyTorch")
    print("Install: pip install torchao")
    TORCHAO_AVAILABLE = False

## 2. Manual INT8 Quantize / Dequantize

Let's implement quantization from scratch to understand the mechanics.

In [ ]:
torch.manual_seed(42)

# Create a realistic weight tensor
weights = torch.randn(4, 8) * 0.5
print("Original FP32 weights:")
print(weights)
print(f"Range: [{weights.min():.4f}, {weights.max():.4f}]")

In [ ]:
# Symmetric INT8 quantization
abs_max = weights.abs().max()
scale = abs_max / 127.0  # INT8 signed range: [-128, 127]

# Quantize: FP32 → INT8
quantized = torch.clamp(torch.round(weights / scale), -128, 127).to(torch.int8)
print("Quantized INT8 values:")
print(quantized)
print(f"Scale: {scale:.6f}")

In [ ]:
# Dequantize: INT8 → FP32 (approximate)
dequantized = quantized.float() * scale
print("Dequantized (FP32 approx):")
print(dequantized)

# Quantization error
error = (weights - dequantized).abs()
print(f"\nMax absolute error: {error.max():.6f}")
print(f"Mean absolute error: {error.mean():.6f}")
print(f"Relative error: {error.mean() / weights.abs().mean():.2%}")

## 3. Precision Loss Demonstration

How much accuracy do we lose at different bit-widths?

In [ ]:
torch.manual_seed(42)
W = torch.randn(256, 256) * 0.02  # Realistic weight scale
x = torch.randn(16, 256) * 0.1    # Input batch
reference = x @ W.T               # FP32 reference output

print(f"{'Bits':>6} | {'Compression':>12} | {'Weight MAE':>12} | {'Output MAE':>12} | {'Relative':>10}")
print("-" * 65)

for bits in [16, 8, 4, 2]:
    qmax = 2 ** (bits - 1) - 1
    qmin = -(2 ** (bits - 1))
    s = W.abs().max() / qmax
    W_q = torch.clamp(torch.round(W / s), qmin, qmax) * s
    out = x @ W_q.T
    w_err = (W - W_q).abs().mean()
    o_err = (reference - out).abs().mean()
    rel = o_err / reference.abs().mean()
    print(f"{bits:>6} | {32/bits:>11.1f}× | {w_err:>12.8f} | {o_err:>12.8f} | {rel:>9.4%}")

## 4. Memory Savings Comparison

The primary motivation for quantization: fitting larger models in limited GPU memory.

In [ ]:
param_counts = {
    "GPT-2 (124M)": 124_000_000,
    "LLaMA-7B": 7_000_000_000,
    "LLaMA-13B": 13_000_000_000,
    "LLaMA-70B": 70_000_000_000,
}

dtypes = {"FP32": 4, "FP16": 2, "INT8": 1, "INT4": 0.5}

print(f"{'Model':>18} | ", end="")
for d in dtypes:
    print(f"{d:>10} | ", end="")
print()
print("-" * 70)

for name, params in param_counts.items():
    print(f"{name:>18} | ", end="")
    for dtype, bpp in dtypes.items():
        size_gb = params * bpp / (1024**3)
        print(f"{size_gb:>7.1f} GB | ", end="")
    print()

**Key insight**: INT4 quantization lets you run a 70B model on a single 80GB GPU (35 GB), where FP16 would require 140 GB (two GPUs).

## 5. Group-Wise Quantization

Per-tensor quantization uses one scale for the entire tensor. Group-wise quantization uses one scale per group of N values, reducing error.

In [ ]:
torch.manual_seed(42)
W = torch.randn(256, 512) * 0.02
# Add per-channel variance (realistic for trained models)
W = W * (torch.rand(256, 1) * 2 + 0.5)

def quantize_symmetric(w, group_size=None):
    """Symmetric INT8 quantization with optional grouping."""
    if group_size is None:
        # Per-tensor
        scale = w.abs().amax() / 127.0
        q = torch.clamp(torch.round(w / scale), -128, 127)
        return q * scale
    else:
        # Per-group
        orig_shape = w.shape
        w_grouped = w.reshape(-1, group_size)
        scales = w_grouped.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / 127.0
        q = torch.clamp(torch.round(w_grouped / scales), -128, 127)
        return (q * scales).reshape(orig_shape)

methods = [
    ("Per-tensor", None),
    ("Group-256", 256),
    ("Group-128", 128),
    ("Group-32", 32),
]

print(f"{'Method':>15} | {'Mean Error':>12} | {'Max Error':>12}")
print("-" * 45)
for name, gs in methods:
    dq = quantize_symmetric(W, gs)
    err = (W - dq).abs()
    print(f"{name:>15} | {err.mean():>12.8f} | {err.max():>12.8f}")

## 6. torchao `quantize_()` API

torchao provides a single function for all quantization: `quantize_(model, config)`. It replaces weight tensors with quantized tensor subclasses in-place.

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, dim=512, hidden=1024, out_dim=256):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

model = SimpleMLP()
x = torch.randn(4, 512)

def model_size_mb(m):
    return sum(p.nelement() * p.element_size() for p in m.parameters()) / (1024**2)

baseline_size = model_size_mb(model)
baseline_out = model(x)
print(f"Baseline size: {baseline_size:.2f} MB")
print(f"Weight dtype: {model.fc1.weight.dtype}")

if TORCHAO_AVAILABLE:
    from torchao import quantize_
    from torchao.quantization import int8_weight_only

    quantize_(model, int8_weight_only())
    q_size = model_size_mb(model)
    q_out = model(x)
    print(f"\nAfter int8_weight_only():")
    print(f"  Size: {q_size:.2f} MB ({baseline_size / q_size:.1f}× compression)")
    print(f"  Weight type: {type(model.fc1.weight).__name__}")
    print(f"  Max output diff: {(baseline_out - q_out).abs().max():.6f}")
else:
    print("\n[torchao not installed — conceptual explanation]")
    print("quantize_(model, int8_weight_only()) would:")
    print("  1. Walk all nn.Linear layers")
    print("  2. Replace .weight with AffineQuantizedTensor (INT8 storage)")
    print("  3. The module stays nn.Linear — only the weight tensor changes")
    est_size = sum(
        p.numel() * 1 if p.ndim >= 2 else p.numel() * 4
        for p in model.parameters()
    ) / (1024**2)
    print(f"  Estimated size: {est_size:.2f} MB ({baseline_size / est_size:.1f}× compression)")

## 7. Weight-Only vs Dynamic Quantization

| | Weight-Only | Dynamic |
|---|---|---|
| **Weights** | INT8 or INT4 | INT8 |
| **Activations** | FP16/BF16 (unchanged) | INT8 (quantized at runtime) |
| **Matmul** | FP16 (dequant then matmul) | INT8 (integer matmul) |
| **Best for** | Memory-bound (batch=1) | Compute-bound (batch>1) |
| **Calibration** | Not needed | Not needed (dynamic) |

**Weight-only** saves memory but still does FP16 matmul.  
**Dynamic** does INT8 matmul (faster) but adds overhead for activation quantization.

In [ ]:
# Demonstrate the difference conceptually
torch.manual_seed(42)
W = torch.randn(128, 128) * 0.02
x = torch.randn(8, 128)

ref = x @ W.T

# Weight-only: quantize W, keep x in float
scale_w = W.abs().amax() / 127.0
W_q = torch.clamp(torch.round(W / scale_w), -128, 127)
W_deq = W_q * scale_w
out_weight_only = x @ W_deq.T  # FP32 matmul with dequantized weights

# Dynamic: quantize both W and x
scale_x = x.abs().amax() / 127.0
x_q = torch.clamp(torch.round(x / scale_x), -128, 127)
# Integer matmul (simulated — real hardware uses INT8 GEMM)
out_int = (x_q.float() @ W_q.float().T) * (scale_x * scale_w)

print(f"Weight-only output error (mean): {(ref - out_weight_only).abs().mean():.6f}")
print(f"Dynamic output error (mean):     {(ref - out_int).abs().mean():.6f}")
print(f"\nDynamic has slightly higher error due to activation quantization,")
print(f"but the INT8 GEMM is much faster on GPU hardware.")

## 8. Semi-Structured Sparsity (2:4)

NVIDIA Ampere+ GPUs support **hardware-accelerated sparse matmul** with the 2:4 pattern: exactly 2 out of every 4 consecutive values must be zero.

```
Dense:   [1.2, 0.5, 3.1, 0.8]    → all 4 values stored
2:4:     [1.2, 0.0, 3.1, 0.0]    → only 2 non-zeros + 2-bit index
                                    → ~2× compression, ~2× speedup
```

In [ ]:
torch.manual_seed(42)
W = torch.randn(8, 16)

# Apply 2:4 sparsity: zero out 2 smallest magnitude per group of 4
W_sparse = W.clone()
for i in range(W.shape[0]):
    for j in range(0, W.shape[1], 4):
        group = W[i, j:j+4].abs()
        _, smallest = group.topk(2, largest=False)
        for idx in smallest:
            W_sparse[i, j + idx] = 0.0

sparsity = (W_sparse == 0).sum().item() / W_sparse.numel()
error = (W - W_sparse).abs().mean()

print(f"Sparsity: {sparsity:.0%}")
print(f"Weight error (MAE): {error:.6f}")
print(f"\nDense bytes (FP16): {W.numel() * 2}")
print(f"Sparse bytes (FP16): {W.numel() // 2 * 2 + W.numel() // 4}")

# Show a row
print(f"\nRow 0 (dense):  {['  0.00' if v == 0 else f'{v:+.2f}' for v in W[0].tolist()]}")
print(f"Row 0 (sparse): {['  0.00' if v == 0 else f'{v:+.2f}' for v in W_sparse[0].tolist()]}")

## 9. Quantization Decision Tree

```
                        What's your goal?
                              │
                 ┌────────────┴────────────┐
                 ▼                          ▼
            Inference                   Training
                 │                          │
       ┌─────────┴──────────┐         ┌────┴────┐
       ▼                    ▼         ▼         ▼
   Memory-bound?      Compute-bound?  BF16     FP8
   (batch=1, LLMs)    (batch>1)     (default) (H100+)
       │                    │
   ┌───┴───┐              INT8
   ▼       ▼             dynamic
  INT4   INT8
  (max    (good
  savings) balance)
```

### Quick Reference

| Scenario | Method | Memory Savings | Accuracy |
|----------|--------|:-:|:-:|
| LLM serving (batch=1) | `int4_weight_only(group_size=128)` | 4× | Good |
| Batch inference (batch>8) | `int8_dynamic_activation_int8_weight()` | 2× | Good |
| H100 inference | `float8_dynamic_activation_float8_weight()` | 2× | Excellent |
| H100 training | `Float8Linear` | 2× | Excellent |
| Mobile deployment | PT2E + XNNPack | 4× | Good |

In [ ]:
def recommend_strategy(task, batch_size, gpu, model_size_b):
    """Simple decision tree for quantization strategy."""
    if task == "training":
        if gpu in ("H100", "MI300"):
            return "FP8 training (Float8Linear)"
        return "BF16 mixed precision (AMP)"

    if batch_size <= 2 or model_size_b >= 7:
        if model_size_b >= 13:
            return "int4_weight_only(group_size=128)"
        return "int8_weight_only()"
    else:
        if gpu in ("H100", "L40S"):
            return "float8_dynamic_activation_float8_weight()"
        return "int8_dynamic_activation_int8_weight()"

scenarios = [
    ("inference", 1, "A100", 70, "70B LLM serving"),
    ("inference", 32, "A100", 1, "Batch inference, 1B model"),
    ("inference", 1, "H100", 7, "7B LLM on H100"),
    ("training", 64, "H100", 13, "Training 13B on H100"),
    ("training", 16, "A100", 7, "Training 7B on A100"),
]

for task, bs, gpu, size, desc in scenarios:
    rec = recommend_strategy(task, bs, gpu, size)
    print(f"{desc:>30s}  →  {rec}")

## 10. torch.compile + Quantized Models

The key advantage of torchao: quantized models **compose natively** with `torch.compile`. Inductor fuses dequantization into the matmul kernel.

```python
# The torchao workflow (3 lines)
from torchao import quantize_
from torchao.quantization import int8_weight_only

quantize_(model, int8_weight_only())              # Step 1: Quantize
model = torch.compile(model, mode="max-autotune")  # Step 2: Compile
output = model(input)                              # Step 3: Run
```

### What Inductor does

```
Without compile: input → dequant(INT8→FP16) → matmul → output  (2 kernels, intermediate FP16 tensor)
With compile:    input → fused_int8_matmul → output             (1 kernel, no intermediate)
```

In [ ]:
# Demonstrate torch.compile with a simple model
model = SimpleMLP(dim=256, hidden=512, out_dim=128)
x = torch.randn(8, 256)

# Eager baseline
t0 = time.perf_counter()
for _ in range(100):
    _ = model(x)
eager_ms = (time.perf_counter() - t0) / 100 * 1000
print(f"Eager FP32: {eager_ms:.2f} ms")

# Compiled
try:
    compiled = torch.compile(model)
    _ = compiled(x)  # warmup / compile
    t0 = time.perf_counter()
    for _ in range(100):
        _ = compiled(x)
    compiled_ms = (time.perf_counter() - t0) / 100 * 1000
    print(f"Compiled FP32: {compiled_ms:.2f} ms")
except Exception as e:
    print(f"torch.compile: {e}")

## 11. Symmetric vs Asymmetric Quantization

- **Symmetric**: Maps `[-abs_max, abs_max]` to `[-128, 127]`. Zero stays at zero. Simpler, no zero_point needed.
- **Asymmetric**: Maps `[min, max]` to `[0, 255]`. Needs a zero_point. Better for skewed distributions (e.g., after ReLU).

In [ ]:
torch.manual_seed(42)

distributions = {
    "Normal (centered)": torch.randn(10000) * 0.5,
    "ReLU output (positive)": torch.relu(torch.randn(10000)) * 0.5,
    "Biased (mean=2)": torch.randn(10000) * 0.5 + 2.0,
}

print(f"{'Distribution':>25} | {'Symmetric Err':>14} | {'Asymmetric Err':>15} | {'Winner':>10}")
print("-" * 75)

for name, data in distributions.items():
    # Symmetric
    s = data.abs().max() / 127.0
    sq = torch.clamp(torch.round(data / s), -128, 127)
    s_err = (data - sq * s).abs().mean()

    # Asymmetric
    a_s = (data.max() - data.min()) / 255.0
    a_zp = torch.round(-data.min() / a_s).clamp(0, 255)
    aq = torch.clamp(torch.round(data / a_s) + a_zp, 0, 255)
    a_err = (data - (aq - a_zp) * a_s).abs().mean()

    winner = "Symmetric" if s_err <= a_err else "Asymmetric"
    print(f"{name:>25} | {s_err:>14.8f} | {a_err:>15.8f} | {winner:>10}")

## 12. Complete Workflow: Quantize and Measure

In [ ]:
# Create a model and measure everything
class BenchModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
        )

    def forward(self, x):
        return self.layers(x)

model = BenchModel()
x = torch.randn(16, 512)

# Measure FP32 baseline
with torch.no_grad():
    ref_out = model(x)

base_params = sum(p.numel() for p in model.parameters())
base_bytes = sum(p.numel() * p.element_size() for p in model.parameters())

print(f"Model: {base_params:,} parameters")
print(f"FP32 size: {base_bytes / 1024**2:.2f} MB")
print(f"Output shape: {ref_out.shape}")

# Simulate quantization at different levels
for bits, label in [(8, "INT8"), (4, "INT4")]:
    m = copy.deepcopy(model)
    with torch.no_grad():
        for param in m.parameters():
            if param.ndim >= 2:
                qmax = 2**(bits-1) - 1
                s = param.abs().amax() / qmax
                param.data = torch.clamp(torch.round(param / s), -qmax-1, qmax) * s
        out = m(x)

    est_bytes = sum(
        p.numel() * bits / 8 if p.ndim >= 2 else p.numel() * 4
        for p in model.parameters()
    )
    diff = (ref_out - out).abs()
    print(f"\n{label}: ~{est_bytes/1024**2:.2f} MB ({base_bytes/est_bytes:.1f}× compression)")
    print(f"  Max output diff: {diff.max():.6f}, Mean: {diff.mean():.6f}")

## 13. Exercise

**Task**: Manually quantize a random weight matrix (256×256) to INT8 using symmetric quantization. Then:

1. Measure the max absolute error
2. Measure the mean absolute error
3. Compute the relative error (mean_abs_error / mean_abs_value)
4. Try group-wise quantization with group_size=64 and compare

**Expected**: Group-wise should reduce error by ~2-4×.

In [ ]:
# YOUR CODE HERE
torch.manual_seed(0)
W = torch.randn(256, 256) * 0.02

# Step 1: Per-tensor symmetric INT8 quantization
# ...

# Step 2: Per-group symmetric INT8 quantization (group_size=64)
# ...

# Step 3: Compare errors
# ...

In [ ]:
# SOLUTION
torch.manual_seed(0)
W = torch.randn(256, 256) * 0.02

# Per-tensor
scale = W.abs().max() / 127.0
W_q = torch.clamp(torch.round(W / scale), -128, 127)
W_dq = W_q * scale
err_pt = (W - W_dq).abs()

print("Per-tensor INT8:")
print(f"  Max error: {err_pt.max():.8f}")
print(f"  Mean error: {err_pt.mean():.8f}")
print(f"  Relative error: {err_pt.mean() / W.abs().mean():.4%}")

# Per-group (group_size=64)
group_size = 64
W_grouped = W.reshape(-1, group_size)
scales = W_grouped.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / 127.0
W_gq = torch.clamp(torch.round(W_grouped / scales), -128, 127)
W_gdq = (W_gq * scales).reshape(256, 256)
err_pg = (W - W_gdq).abs()

print(f"\nPer-group INT8 (group_size={group_size}):")
print(f"  Max error: {err_pg.max():.8f}")
print(f"  Mean error: {err_pg.mean():.8f}")
print(f"  Relative error: {err_pg.mean() / W.abs().mean():.4%}")

print(f"\nImprovement: {err_pt.mean() / err_pg.mean():.2f}× lower error with grouping")

## 14. Key Takeaways

1. **torchao** is the modern PyTorch quantization library — tensor subclass-based, composable with `torch.compile`
2. **`quantize_(model, config)`** is the single API entry point — one line to quantize any model
3. **Weight-only quantization** (INT8/INT4) is best for memory-bound inference (LLM serving at batch=1)
4. **Dynamic quantization** (INT8) is best for compute-bound batch inference
5. **FP8** is the choice for H100+ training and inference — minimal accuracy loss
6. **2:4 sparsity** gives ~2× speedup on NVIDIA Ampere+ with hardware support
7. **torch.compile + torchao** fuses dequantize and matmul into single kernels
8. **Group-wise quantization** reduces error by using per-group scales instead of per-tensor
9. **PT2E** is the path for mobile/edge deployment with backend-specific optimizations

### Next Steps

- Run `quantization_basics.py` for detailed fundamentals
- Run `torchao_workflows.py` for complete benchmarking workflows
- See [Module 29 (Mixed Precision)](../29_mixed_precision/) for FP16/BF16/FP8 training
- See [Module 08 (torch.compile)](../08_torch_compile/) for compilation deep dive

---

<div align="center">

[← Previous: Debugging](../30_debugging/) | [🏠 Home](../README.md) | Next Module →

</div>